[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chihuahualee828/TorchCode/blob/master/solutions/43_llm_training_solution.ipynb)

# 🔴 Solution 42: Train your LLM and chat

Complete assignment 42 first. **Import your own `MiniLLM` from `mini_llm_reference.py`**;
this notebook supplies the data plumbing and ordinary PyTorch training loop.
Implement `LLMTrainer.loss`, `train_step`, and `generate`. Using PyTorch
cross-entropy, AdamW, gradient clipping and DataLoader is explicitly allowed.
Do not call a prebuilt trainer/generation pipeline or load pretrained model weights.

### Data and learning objective
A supplied **UTF-8 byte tokenizer** needs no download or vocabulary fitting and
can represent arbitrary human text. IDs: PAD=0, BOS=1, EOS=2, SEP=3, byte+4.
It uses longer sequences than BPE; implementing BPE from exercise 35 is an extension.
The helper writes 13 original toy training conversations and two separate validation
conversations under `datasets/llm`. It never overwrites existing data.
See the repository's `datasets/llm/README.md` for corpus links and schema details.

A conversation is `[BOS] user: ... [SEP] assistant: ... [EOS]`; earlier assistant
turns end with SEP. The prompt ends at `assistant: `, exactly as during training.
SFT labels mask role prefixes, user/system text and right padding with `-100`,
and supervise assistant content and its end marker. `TextDataset` returns
**unshifted** `(input_ids, labels)`; shift once, inside your loss function.
Text records (`{"text": "..."}`) supervise all next tokens for pretraining.
Long records are right-truncated, and records with no remaining targets are skipped.
For larger datasets, inspect skipped records and use a sufficiently long context.
Splits must happen at the document/conversation level **before** tokenization.

The offline run demonstrates narrow learned replies, not general language
understanding. For broader language skills, pretrain on a much larger text corpus
(e.g. [TinyStories](https://huggingface.co/datasets/roneneldan/TinyStories)), then
continue the **same model/weights** on instruction conversations
(e.g. [smol-smoltalk](https://huggingface.co/datasets/HuggingFaceTB/smol-smoltalk)).
The data README gives download/conversion commands. Those longer runs are optional,
need more compute, and are never downloaded or used by the judge.

### Required methods
- `loss(logits, labels)`: mean CE of `logits[:, :-1]` vs `labels[:, 1:]`, excluding
  `-100`. If all shifted targets are ignored, return a differentiable zero.
- `train_step(model, optimizer, batches, max_norm=1.0)`: each microbatch is an
  `(ids, labels)` pair. Enter training mode, zero gradients once, backpropagate
  the **sum of valid-token losses divided by total valid-token count across all
  microbatches**, clip gradients once, and call optimizer.step once. Return the
  pre-update token-weighted mean loss as a Python float. Reject zero total targets
  with `ValueError` before any update. Different microbatches can have different
  amounts of padding; averaging their individual mean losses is incorrect.
- `generate(model, input_ids, max_new_tokens, eos_id=2)`: batch size one only,
  nonempty prompt, full-prefix inference, eval + no_grad, `argmax` of final logits
  (smallest token ID wins ties). Return prompt plus continuation, including EOS.
  Stop on newly generated EOS, token budget, or context capacity, whichever comes
  first. With budget zero, return an unchanged copy. Reject negative budget or
  overlong/rank-invalid prompts with `ValueError`. Restore the original model
  train/eval mode even if generation raises. Do not mutate the prompt.

### Determinism and evaluation
The judge runs fixed CPU inputs, checks exact shifts/masks and loss gradients,
compares two AdamW updates to a full-batch reference, and checks greedy tokens,
EOS, ties, context stopping and repeatability. It isolates the trainer using small
controlled models; assignment 42 independently grades your real architecture.
The reference and student may produce different trained chat text if their random
initializers differ. Correctness means matching outputs **under identical weights
and inputs**, plus reproducible training and answers for each implementation.
Seeds alone do not make different hardware/software versions bitwise identical.

The runnable experiment below uses one CPU thread, deterministic algorithms,
fixed seed, fixed data order and greedy decoding. Validation measures held-out
loss without updates. It is not a grading threshold for conversational quality.

For an exact same-environment repeatability experiment, rerun `run_training()` and
compare its loss history, model state, and greedy reply. A different valid model
initializer can produce different replies; the grader never expects one universal
answer to open-ended chat. The reference CPU demo learns `Hi` → `Hello!`.


In [ ]:
# Use this repository version: these new tasks may not be on PyPI yet.
# Local: install with `pip install -e /path/to/TorchCode` before launching Jupyter.
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q git+https://github.com/chihuahualee828/TorchCode.git@master')
except ImportError:
    pass

import torch
from torch_judge import check, hint
from torch_judge.capstone import LLMConfig, deterministic

# Run notebook 41 solution in this directory first.
from mini_llm_reference import MiniLLM


In [ ]:
import torch
from torch.nn import functional as F


class LLMTrainer:
    @staticmethod
    def loss(logits, labels):
        targets = labels[:, 1:].reshape(-1)
        if not (targets != -100).any():
            return logits.sum() * 0.0
        return F.cross_entropy(logits[:, :-1].reshape(-1, logits.size(-1)), targets, ignore_index=-100)

    @staticmethod
    def train_step(model, optimizer, batches, max_norm=1.0):
        batches = list(batches)
        count = sum(int((labels[:, 1:] != -100).sum()) for _, labels in batches)
        if count == 0:
            raise ValueError('No supervised targets')
        model.train()
        optimizer.zero_grad(set_to_none=True)
        total = 0.0
        for ids, labels in batches:
            n = int((labels[:, 1:] != -100).sum())
            loss = LLMTrainer.loss(model(ids), labels) * (n / count)
            loss.backward()
            total += loss.item()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm)
        optimizer.step()
        return total

    @staticmethod
    @torch.no_grad()
    def generate(model, input_ids, max_new_tokens, eos_id=2):
        if input_ids.ndim != 2 or input_ids.size(0) != 1 or input_ids.size(1) == 0:
            raise ValueError('Generation expects one nonempty prompt')
        if max_new_tokens < 0 or input_ids.size(1) > model.config.max_seq_len:
            raise ValueError('Invalid generation length')
        was_training = model.training
        model.eval()
        try:
            result = input_ids.clone()
            for _ in range(min(max_new_tokens, model.config.max_seq_len - result.size(1))):
                next_id = model(result)[:, -1].argmax(-1, keepdim=True)
                result = torch.cat((result, next_id), dim=1)
                if next_id.item() == eos_id:
                    break
            return result
        finally:
            model.train(was_training)


In [ ]:
check('llm_training')

### Train from random weights
CPU demonstration; increase epochs to improve memorization. This does not establish generalization.

In [ ]:
from pathlib import Path
from dataclasses import asdict
from torch.utils.data import DataLoader
from torch_judge.capstone import ByteTokenizer, TextDataset, read_jsonl, prepare_demo

DATA = prepare_demo(Path('datasets/llm'))
tokenizer = ByteTokenizer()
config = LLMConfig(d_model=64, num_layers=2, hidden_dim=128, max_seq_len=128)
train_data = TextDataset(read_jsonl(DATA / 'train.jsonl'), config.max_seq_len)
valid_data = TextDataset(read_jsonl(DATA / 'validation.jsonl'), config.max_seq_len)
# Set these paths to your converted corpora for pretraining followed by SFT.
# TextDataset accepts both {"text": ...} and {"messages": [...]} records.
train_loader = DataLoader(train_data, batch_size=4, shuffle=False, num_workers=0)
valid_loader = DataLoader(valid_data, batch_size=4, shuffle=False, num_workers=0)


def evaluate(model, loader):
    was_training = model.training
    model.eval()
    total, count = 0.0, 0
    try:
        with torch.no_grad():
            for ids, labels in loader:
                n = int((labels[:, 1:] != -100).sum())
                total += LLMTrainer.loss(model(ids), labels).item() * n
                count += n
    finally:
        model.train(was_training)
    return total / count


def run_training(epochs=60):
    with deterministic(2026):
        model = MiniLLM(config).cpu()
        # Small tied embeddings improve the from-scratch demo; initialization is provided.
        with torch.no_grad():
            for name, p in sorted(model.named_parameters()):
                if p.ndim >= 2:
                    p.normal_(0.0, 0.02)
        optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3, weight_decay=0.01)
        history = []
        for epoch in range(epochs):
            # One optimizer step per two microbatches, including the final partial group.
            group = []
            for batch in train_loader:
                group.append(batch)
                if len(group) == 2:
                    LLMTrainer.train_step(model, optimizer, group, max_norm=1.0)
                    group = []
            if group:
                LLMTrainer.train_step(model, optimizer, group, max_norm=1.0)
            row = (evaluate(model, train_loader), evaluate(model, valid_loader))
            history.append(row)
            if epoch == 0 or (epoch + 1) % 20 == 0:
                print(f'Epoch {epoch+1}: train={row[0]:.4f}, validation={row[1]:.4f}')
        return model, optimizer, history

model, optimizer, history = run_training()


In [ ]:
# Save primitive config + state dictionaries; no pickled model class.
checkpoint_path = Path('mini_llm_reference_checkpoint.pt')
torch.save({'config': asdict(config), 'model': model.state_dict(),
            'optimizer': optimizer.state_dict(), 'epochs': len(history),
            'seed': 2026, 'tokenizer': 'utf8-byte-v1', 'history': history}, checkpoint_path)
saved = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
restored = MiniLLM(LLMConfig(**saved['config']))
restored.load_state_dict(saved['model'])
restored_optimizer = torch.optim.AdamW(restored.parameters(), lr=3e-3)
restored_optimizer.load_state_dict(saved['optimizer'])
with deterministic(2026):
    probe = torch.tensor([[1, 8, 12]])
    assert torch.equal(model(probe), restored(probe))
# Fixed-order data and no dropout make epoch-boundary continuation reproducible.
# A shuffled/stochastic extension must also save RNG and sampler state.


In [ ]:
def chat_reply(model, user_text, history=None, max_new_tokens=48):
    messages = list(history or []) + [{'role': 'user', 'content': user_text}]
    ids = tokenizer.prompt(messages)
    # Reserve the requested response capacity; drop oldest complete turns first.
    limit = model.config.max_seq_len - max_new_tokens
    while len(ids) > limit and len(messages) > 1:
        messages = messages[2:]
        ids = tokenizer.prompt(messages)
    if len(ids) > limit:
        raise ValueError('Message too long; shorten it or increase the context window')
    prompt = torch.tensor([ids], dtype=torch.long)
    with deterministic(2026):
        output = LLMTrainer.generate(model, prompt, max_new_tokens, tokenizer.eos_id)
    reply = tokenizer.decode(output[0, len(ids):].tolist())
    return reply, messages + [{'role': 'assistant', 'content': reply}]

reply, conversation = chat_reply(restored, 'Hi')
print('User: Hi\nAssistant:', reply)
assert reply == chat_reply(restored, 'Hi')[0]
# Multi-turn use: reply, conversation = chat_reply(restored, 'What is your name?', conversation)
# Interactive loop (opt-in so Run All never blocks):
# conversation = []
# while True:
#     message = input('You (or /quit): ')
#     if message == '/quit':
#         break
#     reply, conversation = chat_reply(restored, message, conversation)
#     print('Assistant:', reply)
